<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Week5/DLP_Week5_lecture_demo_extended.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# !pip install transformers datasets torch

In [3]:
from datasets import load_dataset

In [4]:
ds = load_dataset("facebook/voxpopuli","en", split="train",streaming=True)
from pprint import pprint
pprint(ds)

README.md:   0%|          | 0.00/31.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

IterableDataset({
    features: ['audio_id', 'language', 'audio', 'raw_text', 'normalized_text', 'gender', 'speaker_id', 'is_gold_transcript', 'accent'],
    num_shards: 30
})


In [5]:
subset = []
for i, example in enumerate(ds):
  if i>=200:
    break
  subset.append(example)

In [6]:
ds_es = load_dataset("facebook/voxpopuli","es", split="train",streaming=True)
pprint(ds_es)

IterableDataset({
    features: ['audio_id', 'language', 'audio', 'raw_text', 'normalized_text', 'gender', 'speaker_id', 'is_gold_transcript', 'accent'],
    num_shards: 10
})


In [7]:
for i, example in enumerate(ds_es):
  if i>=200:
    break
  subset.append(example)

In [8]:
ds_ro = load_dataset("facebook/voxpopuli","ro", split="train",streaming=True)
pprint(ds_ro)

IterableDataset({
    features: ['audio_id', 'language', 'audio', 'raw_text', 'normalized_text', 'gender', 'speaker_id', 'is_gold_transcript', 'accent'],
    num_shards: 5
})


In [9]:
for i, example in enumerate(ds_ro):
  if i>=200:
    break
  subset.append(example)

In [10]:
len(subset)

600

In [11]:
subset[0]

{'audio_id': '20180418-0900-PLENARY-3-en_20180418-08:50:36_17',
 'language': 0,
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x79ef6f4792b0>,
 'raw_text': 'If you do not address this problem, the ground is there for populist nationalist forces to go on growing all over Europe.',
 'normalized_text': 'if you do not address this problem the ground is there for populist nationalist forces to go on growing all over europe.',
 'gender': 'female',
 'speaker_id': '124737',
 'is_gold_transcript': True,
 'accent': 'None'}

In [12]:
subset[60]

{'audio_id': '20130116-0900-PLENARY-13-en_20130116-17:48:36_6',
 'language': 0,
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x79ee58594fa0>,
 'raw_text': 'More also needs to be done to fight the endemic corruption that we see in the inspection system and not just in the inspection system, but also in the claims made by the brands, the multinationals, in terms of their corporate responsibility statements.',
 'normalized_text': 'more also needs to be done to fight the endemic corruption that we see in the inspection system and not just in the inspection system but also in the claims made by the brands the multinationals in terms of their corporate responsibility statements.',
 'gender': 'male',
 'speaker_id': '112014',
 'is_gold_transcript': True,
 'accent': 'None'}

In [13]:
subset[111]

{'audio_id': '20180117-0900-PLENARY-16-en_20180117-15:47:35_7',
 'language': 0,
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x79ee58596820>,
 'raw_text': 'You give Mr Putin too much credit.',
 'normalized_text': 'you give mr putin too much credit.',
 'gender': 'male',
 'speaker_id': '28497',
 'is_gold_transcript': True,
 'accent': 'None'}

In [14]:
import random
random.shuffle(subset)

In [15]:
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm

In [16]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
cell_count=0

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def extract_wav2vec_features(batch):
  global cell_count
  cell_count += 1
  print(cell_count)
  inputs = processor(batch["audio"]["array"], sampling_rate=16000, return_tensors="pt", padding=True)
  with torch.no_grad():
    outputs = model(**inputs)
  return {"features": outputs.last_hidden_state.squeeze().mean(dim=0).numpy(), "label": batch["language"]}

In [18]:
subset_with_features = [extract_wav2vec_features(example) for example in subset]

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


In [19]:
from sklearn.model_selection import train_test_split

In [20]:
train_data, test_data = train_test_split(subset_with_features, test_size=0.2)
train_data, val_data= train_test_split(train_data, test_size=0.1)

In [21]:
def prepare_data(data):
  features = [item["features"] for item in data]
  labels = [item["label"] for item in data]
  return torch.tensor(features), torch.tensor(labels)

In [22]:
train_features, train_labels = prepare_data(train_data)
val_features, val_labels = prepare_data(val_data)
test_features, test_labels = prepare_data(test_data)

/tmp/ipykernel_192/3983969988.py:4: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  return torch.tensor(features), torch.tensor(labels)


In [23]:
# building a classifier using CNN
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class CNNModel(nn.Module):
  def __init__(self, input_dim, num_classes):
    super(CNNModel, self).__init__()
    self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=128, kernel_size=3, padding=1)
    self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
    self.fc1 = nn.Linear(256*(input_length//4), 128)
    self.fc2 = nn.Linear(128, num_classes)

  def forward(self,x):
    x = F.relu(self.conv1(x))
    x = F.max_pool1d(x, kernel_size=2)
    x = F.relu(self.conv2(x))
    x = F.max_pool1d(x, kernel_size=2)
    x = x.view(x.size(0), -1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

In [24]:
input_length = train_features.shape[1]

In [25]:
label_mapping = {6:0, 0:1, 3:2}

In [26]:
train_labels = train_labels.apply_(lambda x: label_mapping[x])
val_labels = val_labels.apply_(lambda x: label_mapping[x])
test_labels = test_labels.apply_(lambda x: label_mapping[x])

In [27]:
unique_train_labels = torch.unique(train_labels)
unique_test_labels = torch.unique(test_labels)
unique_val_labels = torch.unique(val_labels)
print(f"Unique Train labels: {unique_train_labels}")
print(f"Unique Val labels: {unique_val_labels}")
print(f"Unique Test labels: {unique_test_labels}")

Unique Train labels: tensor([0, 1, 2])
Unique Val labels: tensor([0, 1, 2])
Unique Test labels: tensor([0, 1, 2])


In [28]:
model = CNNModel(input_dim=1, num_classes=3)

In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [30]:
num_epochs = 100
for epoch in range(num_epochs):
  model.train()
  optimizer.zero_grad()
  outputs = model(train_features.unsqueeze(1).float())
  loss = criterion(outputs, train_labels)
  loss.backward()
  optimizer.step()

  # validation
  model.eval()
  with torch.no_grad():
    val_outputs = model(val_features.unsqueeze(1).float())
    val_loss = criterion(val_outputs, val_labels)
  print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}, Val Loss: {val_loss.item()}")

Epoch 1/100, Loss: 1.0988010168075562, Val Loss: 1.8639205694198608
Epoch 2/100, Loss: 2.236090898513794, Val Loss: 1.030829906463623
Epoch 3/100, Loss: 1.0723860263824463, Val Loss: 1.017027735710144
Epoch 4/100, Loss: 1.0422149896621704, Val Loss: 0.9588932991027832
Epoch 5/100, Loss: 1.0055361986160278, Val Loss: 0.9579079747200012
Epoch 6/100, Loss: 0.9801784753799438, Val Loss: 0.8763210773468018
Epoch 7/100, Loss: 0.9223305583000183, Val Loss: 0.8073859810829163
Epoch 8/100, Loss: 0.8879759907722473, Val Loss: 0.7756197452545166
Epoch 9/100, Loss: 0.8312491178512573, Val Loss: 0.6799132823944092
Epoch 10/100, Loss: 0.7604847550392151, Val Loss: 0.6332586407661438
Epoch 11/100, Loss: 0.7325623035430908, Val Loss: 0.6469898819923401
Epoch 12/100, Loss: 0.7523281574249268, Val Loss: 0.5582215189933777
Epoch 13/100, Loss: 0.6278536915779114, Val Loss: 0.5843797922134399
Epoch 14/100, Loss: 0.6679731011390686, Val Loss: 0.4589918851852417
Epoch 15/100, Loss: 0.5712538957595825, Val Lo

In [31]:
from sklearn.metrics import accuracy_score

In [32]:
model.eval()
with torch.no_grad():
  test_outputs = model(test_features.unsqueeze(1).float())
  _, predicted = torch.max(test_outputs, 1)
  accuracy = accuracy_score(predicted.numpy(), test_labels.numpy())
print(f"Test Accuracy: {accuracy*100:.3f}%")

Test Accuracy: 85.833%
